# Bio-PM IRB V3: Data-Driven Analysis Notebook

This notebook is intentionally factual-only:
- No hardcoded outcome claims
- Every metric is computed from local artifacts
- If a file is missing, the notebook reports it explicitly


In [11]:
import os, glob, json
import numpy as np
import h5py

try:
    import pandas as pd
except Exception:
    pd = None

PIPELINES = [
    ("standard", "preprocessed", "features/biopm_features.npz"),
    ("alt", "preprocessed_alt", "features/biopm_features_alt.npz"),
    ("v3", "preprocessed_v3", "features/biopm_features_v3.npz"),
]
LEGACY_PATHS = [
    "features/biopm_features_legacy_schema.npz",
    "features/biopm_features_legacy_schema_alt.npz",
    "features/biopm_features_legacy_schema_v3.npz",
]
print('Configured pipelines:')
for p in PIPELINES:
    print(' ', p)


Configured pipelines:
  ('standard', 'preprocessed', 'features/biopm_features.npz')
  ('alt', 'preprocessed_alt', 'features/biopm_features_alt.npz')
  ('v3', 'preprocessed_v3', 'features/biopm_features_v3.npz')


In [12]:
def h5_stats(root):
    files = sorted(glob.glob(os.path.join(root, '*.h5')))
    if not files:
        return dict(files=0, windows=0, fill_global_pct=np.nan, avg_me=np.nan, med_me=np.nan, p95_me=np.nan, grav_t=[], me_min=np.nan, me_max=np.nan)

    total_valid = 0
    total_slots = 0
    total_windows = 0
    me_counts = []
    grav_t = []

    for fp in files:
        with h5py.File(fp, 'r') as hf:
            xa = np.array(hf['x_acc_filt'])
            xg = np.array(hf['x_gravity'])

        valid = ~np.isnan(xa[:, :, 0])
        total_valid += int(valid.sum())
        total_slots += int(valid.size)
        total_windows += int(valid.shape[0])
        me_counts.append(valid.sum(axis=1))
        grav_t.append(int(xg.shape[1]))

    me = np.concatenate(me_counts)
    return dict(
        files=len(files),
        windows=total_windows,
        fill_global_pct=100.0 * total_valid / total_slots,
        avg_me=float(me.mean()),
        med_me=float(np.median(me)),
        p95_me=float(np.percentile(me, 95)),
        me_min=float(me.min()),
        me_max=float(me.max()),
        grav_t=sorted(set(grav_t)),
    )

def feature_stats(path):
    if not os.path.exists(path):
        return dict(exists=False, feature_shape='missing', n_windows=np.nan, n_dim=np.nan, mean_abs=np.nan, std_abs=np.nan, grav_abs=np.nan, healthy_pct=np.nan, n_subjects=np.nan, n_zero_rows=np.nan)

    d = np.load(path, allow_pickle=True)
    X = d['features']
    y = d['labels']
    subj = d['subj'] if 'subj' in d else d['pids']

    return dict(
        exists=True,
        feature_shape=str(tuple(X.shape)),
        n_windows=int(X.shape[0]),
        n_dim=int(X.shape[1]),
        mean_abs=float(np.abs(X[:, :64]).mean()),
        std_abs=float(np.abs(X[:, 64:128]).mean()),
        grav_abs=float(np.abs(X[:, 128:]).mean()),
        healthy_pct=float((y == 1).mean() * 100.0),
        n_subjects=int(len(np.unique(subj))),
        n_zero_rows=int((np.abs(X) < 1e-6).all(axis=1).sum()),
    )

def show_table(rows, order=None, title=None):
    if title:
        print('\n' + title)
    if pd is None:
        for r in rows:
            print(r)
        return
    df = pd.DataFrame(rows)
    if order is not None:
        cols = [c for c in order if c in df.columns] + [c for c in df.columns if c not in order]
        df = df[cols]
    num_cols = df.select_dtypes(include=['number']).columns
    for c in num_cols:
        df[c] = df[c].map(lambda v: v if pd.isna(v) else round(float(v), 6))
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df.to_string(index=False))


## 1) HDF5 Fill-Rate and Token Occupancy Audit
Computed directly from each `x_acc_filt` tensor in each pipeline directory.


In [13]:
h5_rows = []
for name, h5_root, _ in PIPELINES:
    row = {'pipeline': name, 'h5_root': h5_root, 'h5_exists': os.path.isdir(h5_root)}
    if os.path.isdir(h5_root):
        row.update(h5_stats(h5_root))
    else:
        row.update(dict(files=0, windows=0, fill_global_pct=np.nan, avg_me=np.nan, med_me=np.nan, p95_me=np.nan, me_min=np.nan, me_max=np.nan, grav_t=[]))
    h5_rows.append(row)

show_table(
    h5_rows,
    order=['pipeline','h5_root','h5_exists','files','windows','grav_t','fill_global_pct','avg_me','med_me','p95_me','me_min','me_max'],
    title='HDF5 audit results'
)



HDF5 audit results


,pipeline,h5_root,h5_exists,files,windows,grav_t,fill_global_pct,avg_me,med_me,p95_me,me_min,me_max
0,standard,preprocessed,True,198.0,587046.0,[90],31.660405,18.046431,18.0,26.0,3.0,47.0
1,alt,preprocessed_alt,True,198.0,195610.0,[270],34.673433,59.291570,59.0,79.0,20.0,139.0
2,v3,preprocessed_v3,True,198.0,195610.0,[270],94.066120,53.617688,57.0,57.0,20.0,57.0


## 2) Feature File Audit
Computed directly from generated `.npz` feature files.


In [14]:
feat_rows = []
for name, _, feat_path in PIPELINES:
    row = {'pipeline': name, 'features_path': feat_path}
    row.update(feature_stats(feat_path))
    feat_rows.append(row)

show_table(
    feat_rows,
    order=['pipeline','features_path','exists','feature_shape','n_windows','n_dim','mean_abs','std_abs','grav_abs','healthy_pct','n_subjects','n_zero_rows'],
    title='Feature audit results'
)



Feature audit results


,pipeline,features_path,exists,feature_shape,n_windows,n_dim,mean_abs,std_abs,grav_abs,healthy_pct,n_subjects,n_zero_rows
0,standard,features/biopm_features.npz,True,"(587046, 1028)",587046.0,1028.0,0.709126,0.418573,0.055206,4.835907,36.0,0.0
1,alt,features/biopm_features_alt.npz,True,"(195610, 1028)",195610.0,1028.0,0.698720,0.435994,0.055810,4.837176,36.0,0.0
2,v3,features/biopm_features_v3.npz,True,"(195610, 1028)",195610.0,1028.0,0.742892,0.365872,0.095381,4.837176,36.0,0.0


## 3) Consistency Checks (HDF5 vs Features)
Checks whether total windows in HDF5 match feature row counts per pipeline.


In [15]:
checks = []
for h, f in zip(h5_rows, feat_rows):
    h_n = h.get('windows', np.nan)
    f_n = f.get('n_windows', np.nan)
    ok = (not np.isnan(h_n)) and (not np.isnan(f_n)) and (int(h_n) == int(f_n))
    checks.append({
        'pipeline': h['pipeline'],
        'h5_windows': h_n,
        'feature_rows': f_n,
        'match': bool(ok)
    })

show_table(checks, order=['pipeline','h5_windows','feature_rows','match'], title='Window-count consistency')



Window-count consistency


,pipeline,h5_windows,feature_rows,match
0,standard,587046.0,587046.0,True
1,alt,195610.0,195610.0,True
2,v3,195610.0,195610.0,True


## 4) Legacy Schema File Check
Confirms existence and key shapes for visit-level legacy exports.


In [16]:
legacy_rows = []
for p in LEGACY_PATHS:
    row = {'legacy_path': p, 'exists': os.path.exists(p)}
    if row['exists']:
        d = np.load(p, allow_pickle=True)
        row['features_shape'] = str(tuple(d['features'].shape)) if 'features' in d else 'missing'
        row['features_even_shape'] = str(tuple(d['features_even'].shape)) if 'features_even' in d else 'missing'
        row['features_odd_shape'] = str(tuple(d['features_odd'].shape)) if 'features_odd' in d else 'missing'
        row['labels_shape'] = str(tuple(d['labels'].shape)) if 'labels' in d else 'missing'
    legacy_rows.append(row)

show_table(legacy_rows, order=['legacy_path','exists','features_shape','features_even_shape','features_odd_shape','labels_shape'], title='Legacy schema audit')



Legacy schema audit


,legacy_path,exists,features_shape,features_even_shape,features_odd_shape,labels_shape
0,features/biopm_features_legacy_schema.npz,True,"(198, 1028)","(198, 1028)","(198, 1028)","(198,)"
1,features/biopm_features_legacy_schema_alt.npz,True,"(198, 1028)","(198, 1028)","(198, 1028)","(198,)"
2,features/biopm_features_legacy_schema_v3.npz,True,"(198, 1028)","(198, 1028)","(198, 1028)","(198,)"


## 5) Delta View (V3 vs Standard / Alt)
Numeric deltas only; no interpretation text is hardcoded.


In [17]:
if pd is None:
    print('pandas not available; skipping delta table.')
else:
    hdf = pd.DataFrame(h5_rows).set_index('pipeline')
    fdf = pd.DataFrame(feat_rows).set_index('pipeline')

    metrics = ['fill_global_pct','avg_me','mean_abs','std_abs','grav_abs']
    rows = []
    for base in ['standard','alt']:
        if 'v3' in hdf.index and base in hdf.index and 'v3' in fdf.index and base in fdf.index:
            row = {'delta': f'v3 - {base}'}
            row['fill_global_pct'] = float(hdf.loc['v3','fill_global_pct'] - hdf.loc[base,'fill_global_pct'])
            row['avg_me'] = float(hdf.loc['v3','avg_me'] - hdf.loc[base,'avg_me'])
            row['mean_abs'] = float(fdf.loc['v3','mean_abs'] - fdf.loc[base,'mean_abs'])
            row['std_abs'] = float(fdf.loc['v3','std_abs'] - fdf.loc[base,'std_abs'])
            row['grav_abs'] = float(fdf.loc['v3','grav_abs'] - fdf.loc[base,'grav_abs'])
            rows.append(row)

    show_table(rows, order=['delta'] + metrics, title='Delta table')



Delta table


,delta,fill_global_pct,avg_me,mean_abs,std_abs,grav_abs
0,v3 - standard,62.405715,35.571257,0.033766,-0.052701,0.040176
1,v3 - alt,59.392687,-5.673882,0.044172,-0.070122,0.039572
